# zarr-java OME-Zarr — Integration Tests

Verifies that the code examples from `USERGUIDE-OME-ZARR.md` work against the locally-built JARs.

**Before opening this notebook**, launch it via `./notebooks/start.sh` from the repo root.  
That script builds the project, sets up the classpath, and starts JupyterLab — no internet access needed for the library.

Run cells top-to-bottom with **Run All Cells** or `Shift+Enter`.

## Setup

In [1]:
// Confirm the library and repo root are available (set by start.sh)
String REPO_ROOT = System.getenv("REPO_ROOT");
if (REPO_ROOT == null) throw new IllegalStateException(
    "REPO_ROOT is not set. Please start JupyterLab via ./notebooks/start.sh");
System.out.println("Repo root : " + REPO_ROOT);
System.out.println("Library   : zarr-java-ome (loaded via IJAVA_CLASSPATH)");

Repo root : /home/konstantin/Documents/zarr-java
Library   : zarr-java-ome (loaded via IJAVA_CLASSPATH)


## 1 · Open a v0.5 multiscale image

In [2]:
import dev.zarr.omezarr.MultiscaleImage;
import dev.zarr.zarrjava.store.FilesystemStore;
import dev.zarr.zarrjava.store.StoreHandle;
import dev.zarr.omezarr.metadata.MultiscalesEntry;

StoreHandle imageHandle = new FilesystemStore(REPO_ROOT + "/zarr-java-ome/testdata/ome/v0.5").resolve();
MultiscaleImage image = MultiscaleImage.open(imageHandle);

System.out.println("Scale levels : " + image.getScaleLevelCount());
System.out.println("Axis names   : " + image.getAxisNames());
System.out.println("Dataset[0]   : " + image.getMultiscaleNode(0).datasets.get(0).path);

Scale levels : 2
Axis names   : [t, c, z, y, x]
Dataset[0]   : 0


## 2 · Read a scale level array

In [3]:
dev.zarr.zarrjava.core.Array s0 = image.openScaleLevel(0);
ucar.ma2.Array full   = s0.read();
ucar.ma2.Array subset = s0.read(new long[]{0,0,0,0,0}, new long[]{1,1,4,8,8});

System.out.println("Full shape  : " + java.util.Arrays.toString(full.getShape()));
System.out.println("Subset shape: " + java.util.Arrays.toString(subset.getShape()));

Full shape  : [1, 2, 8, 16, 16]
Subset shape: [1, 1, 4, 8, 8]


## 3 · Labels sub-group

In [4]:
java.util.List<String> labels = image.getLabels();
System.out.println("Labels: " + labels);

MultiscaleImage labelImage = image.openLabel(labels.get(0));
System.out.println("Label axis names: " + labelImage.getAxisNames());

Labels: [nuclei]
Label axis names: [z, y, x]


## 4 · HCS plate and well (v0.5)

In [5]:
import dev.zarr.omezarr.Plate;
import dev.zarr.omezarr.Well;

StoreHandle plateHandle = new FilesystemStore(REPO_ROOT + "/zarr-java-ome/testdata/ome/v0.5_hcs").resolve();
Plate plate = Plate.open(plateHandle);
Well well   = plate.openWell("A/1");
MultiscaleImage wellImage = well.openImage("0");

System.out.println("Well axis names: " + wellImage.getAxisNames());

Well axis names: [t, c, z, y, x]


## 5 · Write: create a new v0.5 multiscale image

In [6]:
import dev.zarr.omezarr.metadata.Axis;
import dev.zarr.omezarr.metadata.Dataset;
import dev.zarr.omezarr.metadata.transform.CoordinateTransformation;
import dev.zarr.zarrjava.v3.Array;
import dev.zarr.zarrjava.v3.DataType;
import java.util.Arrays;
import java.util.Collections;

new ProcessBuilder("rm", "-rf", "/tmp/ome_test_v05.zarr").start().waitFor();

StoreHandle out = new FilesystemStore("/tmp/ome_test_v05.zarr").resolve();
MultiscalesEntry ms = new MultiscalesEntry(
    Arrays.asList(new Axis("y", "space", "micrometer"), new Axis("x", "space", "micrometer")),
    Collections.<Dataset>emptyList()
);
dev.zarr.omezarr.v0_5.MultiscaleImage written =
    dev.zarr.omezarr.v0_5.MultiscaleImage.create(out, ms);

written.createScaleLevel(
    "s0",
    Array.metadataBuilder().withShape(1024, 1024).withChunkShape(256, 256).withDataType(DataType.UINT16).build(),
    Collections.singletonList(CoordinateTransformation.scale(Arrays.asList(1.0, 1.0)))
);
written.createScaleLevel(
    "s1",
    Array.metadataBuilder().withShape(512, 512).withChunkShape(256, 256).withDataType(DataType.UINT16).build(),
    Collections.singletonList(CoordinateTransformation.scale(Arrays.asList(2.0, 2.0)))
);
System.out.println("Written to /tmp/ome_test_v05.zarr");

Written to /tmp/ome_test_v05.zarr


## 6 · Re-open and verify the written image

In [7]:
MultiscaleImage reopened = MultiscaleImage.open(out);
MultiscalesEntry reopenedMs = reopened.getMultiscaleNode(0);
var s1Tx = (dev.zarr.omezarr.metadata.transform.ScaleCoordinateTransformation)
    reopenedMs.datasets.get(1).coordinateTransformations.get(0);

System.out.println("Is v0.5 instance : " + (reopened instanceof dev.zarr.omezarr.v0_5.MultiscaleImage));
System.out.println("Scale level count: " + reopened.getScaleLevelCount());
System.out.println("s0 path          : " + reopenedMs.datasets.get(0).path);
System.out.println("s1 path          : " + reopenedMs.datasets.get(1).path);
System.out.println("s1 scale         : " + s1Tx.scale);

Is v0.5 instance : true
Scale level count: 2
s0 path          : s0
s1 path          : s1
s1 scale         : [2.0, 2.0]


## 7 · v0.4 image (Zarr v2 layout)

In [8]:
StoreHandle v04Handle = new FilesystemStore(REPO_ROOT + "/zarr-java-ome/testdata/ome/v0.4").resolve();
MultiscaleImage v04 = MultiscaleImage.open(v04Handle);

System.out.println("v0.4 scale levels: " + v04.getScaleLevelCount());
System.out.println("v0.4 axis names  : " + v04.getAxisNames());
System.out.println("v0.4 typed       : " + (v04 instanceof dev.zarr.omezarr.v0_4.MultiscaleImage));

v0.4 scale levels: 2
v0.4 axis names  : [t, c, z, y, x]
v0.4 typed       : true


## 8 · v0.4 HCS plate and well

In [9]:
StoreHandle hcs04 = new FilesystemStore(REPO_ROOT + "/zarr-java-ome/testdata/ome/v0.4_hcs").resolve();
Plate plate04 = Plate.open(hcs04);
Well well04   = plate04.openWell("A/1");
MultiscaleImage wellImage04 = well04.openImage("0");

System.out.println("v0.4 HCS well axis names: " + wellImage04.getAxisNames());

v0.4 HCS well axis names: [t, c, z, y, x]
